# Notebook 1: Data Loading & Preprocessing
## BFI-Based Few-Shot Binary Occupancy Detection

This notebook processes raw `.npy` V-matrix files (output of Wi-BFI) into
clean, windowed tensors — **one output file per input trace**.

**What this notebook does:**
1. Loads every V-matrix `.npy` trace, keeping each file as a separate record
2. Truncates to common dimensions (234 subcarriers, 1 spatial stream)
3. Separates real and imaginary parts into CNN-ready channels
4. Diagnoses and handles the structurally-zero Im(ant2) channel
5. Filters out bad frames (NaN, inf, zero-only) per trace
6. Applies per-trace, per-channel z-score normalization
7. Segments each trace into non-overlapping temporal windows independently (no cross-trace contamination)
8. Saves one `p_{original_stem}.npy` file per trace — label is re-derived from the filename at load time

Each trace retains its own window count W after step 7. There is no global frame cap
or cross-trace resampling — cross-device imbalance is handled at training time via
stratified episode sampling in Notebook 4.

**No PyTorch here.** Pure NumPy.

**Output naming convention:**
```
data/processed/
  {device}/
    p_vmatrix_empty_M7.npy
    p_vmatrix_stationary_M7.npy
    p_vmatrix_moving_M7.npy
    ...
```

> **Note for Notebooks 2–4:** Saved arrays have shape `(W, T, K, C)` — channels last.
> PyTorch CNNs expect channels first. Apply this when loading:
> ```python
> x = torch.from_numpy(np.load(path)).float().permute(0, 3, 1, 2)
> # (W, T, K, C) → (W, C, T, K)
> ```

---

## 0. Configuration

**Edit this cell to match your actual file paths and folder structure.**

Expected folder layout:
```
data/
  M7/
    vmatrix_empty_M7.npy
    vmatrix_stationary_M7.npy
    vmatrix_moving_M7.npy
  X7/ ...
  X300/ ...
```

Each file has shape `(P, K, M, Nss)` where:
- `P` = frames in this capture
- `K` = subcarriers (234 for Wi-Fi 5 / 802.11ac, 250 for Wi-Fi 6 / 802.11ax)
- `M` = TX antennas at the AP (3 for all devices)
- `Nss` = spatial streams (1 for M7, 2 for X7 and X300)

Shapes before truncation:
- POCO M7 (Wi-Fi 5): `(P, 234, 3, 1)`
- POCO X7 (Wi-Fi 6): `(P, 250, 3, 2)`
- Vivo X300 (Wi-Fi 6): `(P, 250, 3, 2)`

In [1]:
import numpy as np
import os
from pathlib import Path

DATA_ROOT = Path("data")  # Root folder containing device subfolders

# Maps device name -> subfolder under DATA_ROOT.
DEVICES = {
    "M7": "M7",
    "X7": "X7",
    "X300": "X300",
}

# Scenario label mapping — matched against each filename stem at load time.
LABEL_MAP = {
    "empty": 0,
    "stationary": 1,
    "moving": 1,
}

# ── Preprocessing parameters ─────────────────────────────────────────────────

TARGET_SUBCARRIERS = 234  # 802.11ac = 234, 802.11ax = 250 → truncate to 234
TARGET_NSS = 1            # M7: Nss=1; X7/X300: Nss=2 → keep stream 0 only

# WINDOW_SIZE = 5 frames per window.
# T=5 is the smallest window size that preserves meaningful temporal structure
# through all three convolutional pooling layers of the encoder while still
# yielding sufficient non-overlapping windows even for the most data-scarce
# sessions. This provides enough windows to support K-shot adaptation with
# multiple query examples per class and stable aggregate metrics over repeated
# random support draws, under the constraints of the synchronized multi-device
# capture protocol.
WINDOW_SIZE = 5

OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Configuration loaded.")
print(f"  Data root : {DATA_ROOT.resolve()}")
folders = [p.name for p in DATA_ROOT.iterdir() if p.is_dir()] if DATA_ROOT.exists() else []
print(f"  Folders   : {folders}")
print(f"  Frame shape: ({TARGET_SUBCARRIERS}, 3, {TARGET_NSS})")
print(f"  Window size: {WINDOW_SIZE} frames")
print(f"  Output dir : {OUTPUT_DIR}")
print()
print("NOTE: No global frame cap is applied. Each trace keeps its full")
print("post-quality-filter frame count. Cross-device imbalance is handled")
print("at training time via stratified episode sampling (Notebook 4).")

Configuration loaded.
  Data root : C:\Users\ghosty\Desktop\FSL-BFI\data
  Folders   : ['M7', 'processed', 'X300', 'X7']
  Frame shape: (234, 3, 1)
  Window size: 5 frames
  Output dir : data\processed

NOTE: No global frame cap is applied. Each trace keeps its full
post-quality-filter frame count. Cross-device imbalance is handled
at training time via stratified episode sampling (Notebook 4).


---
## 1. Load and Inspect Raw Data

Every `.npy` file is loaded as an independent **trace record** — a dict containing
the original filename stem, device name, scenario, and the raw array.

In [2]:
trace_records = []  # list of dicts, one entry per .npy file

print("Loading raw V-matrix files...")
print(f"{'path':50s} {'shape':25s} {'dtype'}")
print("-" * 85)

prev_device = None
for device_name, folder_name in DEVICES.items():
    folder = DATA_ROOT / folder_name

    if not folder.is_dir():
        print(f"  WARNING: Folder not found: {folder}")
        continue

    npy_files = sorted(folder.glob("*.npy"))
    if not npy_files:
        print(f"  WARNING: No .npy files found in {folder}")
        continue

    if prev_device is not None:
        print()
    prev_device = device_name

    for filepath in npy_files:
        stem = filepath.stem
        scenario = next((k for k in LABEL_MAP if k in stem.lower()), None)

        if scenario is None:
            print(f"  WARNING: Cannot infer scenario from {filepath.name}. Skipping.")
            continue

        v = np.load(filepath)
        trace_records.append({
            "stem": stem,
            "device": device_name,
            "scenario": scenario,
            "data": v,
        })
        print(f"  {(folder_name + '/' + stem + '.npy'):50s} {str(v.shape):25s} {v.dtype}")

print("-" * 85)
print(f"Loaded {len(trace_records)} traces total.")
for dev in DEVICES:
    n = len([r for r in trace_records if r["device"] == dev])
    print(f"  {dev:12s} {n} traces")

if len(trace_records) == 0:
    raise FileNotFoundError("No files loaded! Check that DATA_ROOT points to the correct folder.")

Loading raw V-matrix files...
path                                               shape                     dtype
-------------------------------------------------------------------------------------
  M7/vmatrix_empty-2_M7.npy                          (361, 234, 3, 1)          complex128
  M7/vmatrix_empty-3_M7.npy                          (361, 234, 3, 1)          complex128
  M7/vmatrix_empty_M7.npy                            (371, 234, 3, 1)          complex128
  M7/vmatrix_moving-2_M7.npy                         (375, 234, 3, 1)          complex128
  M7/vmatrix_moving-3_M7.npy                         (335, 234, 3, 1)          complex128
  M7/vmatrix_moving_M7.npy                           (335, 234, 3, 1)          complex128
  M7/vmatrix_stationary-2_M7.npy                     (355, 234, 3, 1)          complex128
  M7/vmatrix_stationary-3_M7.npy                     (366, 234, 3, 1)          complex128
  M7/vmatrix_stationary_M7.npy                       (386, 234, 3, 1)          co

---
## 2. Truncate to Common Dimensions

Truncate every trace to `(P, 234, 3, 1)`:
- First 234 subcarriers — drops 16 edge subcarriers from Wi-Fi 6 devices (250→234)
- First spatial stream only (index 0) — drops Nss=2 from X7 and X300

This gives a uniform representation across all three devices.
Stream 0 is the dominant spatial mode by IEEE 802.11 Givens-rotation convention.

In [3]:
print("Truncating to common dimensions...")
print(f"  Target per frame: ({TARGET_SUBCARRIERS}, 3, {TARGET_NSS})")
print("-" * 70)

for r in trace_records:
    original_shape = r["data"].shape
    r["data"] = r["data"][:, :TARGET_SUBCARRIERS, :, :TARGET_NSS]
    print(f"  {r['device']:8s} {r['scenario']:12s} {r['stem'][-20:]:22s} "
          f"{str(original_shape):22s} -> {r['data'].shape}")

print("-" * 70)
print("All traces truncated.")

Truncating to common dimensions...
  Target per frame: (234, 3, 1)
----------------------------------------------------------------------
  M7       empty        vmatrix_empty-2_M7     (361, 234, 3, 1)       -> (361, 234, 3, 1)
  M7       empty        vmatrix_empty-3_M7     (361, 234, 3, 1)       -> (361, 234, 3, 1)
  M7       empty        vmatrix_empty_M7       (371, 234, 3, 1)       -> (371, 234, 3, 1)
  M7       moving       vmatrix_moving-2_M7    (375, 234, 3, 1)       -> (375, 234, 3, 1)
  M7       moving       vmatrix_moving-3_M7    (335, 234, 3, 1)       -> (335, 234, 3, 1)
  M7       moving       vmatrix_moving_M7      (335, 234, 3, 1)       -> (335, 234, 3, 1)
  M7       stationary   trix_stationary-2_M7   (355, 234, 3, 1)       -> (355, 234, 3, 1)
  M7       stationary   trix_stationary-3_M7   (366, 234, 3, 1)       -> (366, 234, 3, 1)
  M7       stationary   matrix_stationary_M7   (386, 234, 3, 1)       -> (386, 234, 3, 1)
  X7       empty        vmatrix_empty-2_X7     (364,

---
## 3. Extract Real & Imaginary Channels

The V-matrix is complex128. CNNs need real-valued inputs.
We squeeze the Nss=1 axis and concatenate real & imaginary parts:

```
(P, 234, 3, 1) --squeeze--> (P, 234, 3) --real/imag--> (P, 234, 6)
```

Channel layout: Re(ant0), Re(ant1), Re(ant2), Im(ant0), Im(ant1), Im(ant2)

Channel 5 `Im(ant2)` may be structurally zero due to the IEEE 802.11
Givens-rotation BFI compression. Check Section 3.1 before proceeding.

In [4]:
print("Extracting real and imaginary parts...")
print("-" * 70)

for r in trace_records:
    v = r["data"]          # (P, 234, 3, 1)
    vsq = v.squeeze(axis=-1)  # (P, 234, 3)
    features = np.concatenate(
        [vsq.real, vsq.imag], axis=-1
    )  # (P, 234, 6)
    assert features.dtype in (np.float64, np.float32), \
        f"Unexpected dtype: {features.dtype}"
    r["data"] = features
    print(f"  {r['device']:8s} {r['scenario']:12s} {r['stem'][-20:]:22s} "
          f"complex (P,234,3,1) -> real {features.shape}")

print("-" * 70)
print("Channel layout: Re(ant0), Re(ant1), Re(ant2), Im(ant0), Im(ant1), Im(ant2)")

Extracting real and imaginary parts...
----------------------------------------------------------------------
  M7       empty        vmatrix_empty-2_M7     complex (P,234,3,1) -> real (361, 234, 6)
  M7       empty        vmatrix_empty-3_M7     complex (P,234,3,1) -> real (361, 234, 6)
  M7       empty        vmatrix_empty_M7       complex (P,234,3,1) -> real (371, 234, 6)
  M7       moving       vmatrix_moving-2_M7    complex (P,234,3,1) -> real (375, 234, 6)
  M7       moving       vmatrix_moving-3_M7    complex (P,234,3,1) -> real (335, 234, 6)
  M7       moving       vmatrix_moving_M7      complex (P,234,3,1) -> real (335, 234, 6)
  M7       stationary   trix_stationary-2_M7   complex (P,234,3,1) -> real (355, 234, 6)
  M7       stationary   trix_stationary-3_M7   complex (P,234,3,1) -> real (366, 234, 6)
  M7       stationary   matrix_stationary_M7   complex (P,234,3,1) -> real (386, 234, 6)
  X7       empty        vmatrix_empty-2_X7     complex (P,234,3,1) -> real (364, 234, 6)


### 3.1 Diagnostic: Is Im(ant2) Structurally Zero?

IEEE 802.11 BFI encodes V-matrices using Givens rotations. By standard convention,
the last antenna column is kept real (imaginary = 0) as the phase reference.
If channel 5 is always zero it carries no information and should be dropped —
keeping it wastes one CNN filter per layer.

In [5]:
max_abs_imag_ant2 = 0.0
for r in trace_records:
    ch5 = r["data"][:, :, 5]
    max_abs_imag_ant2 = max(max_abs_imag_ant2, np.abs(ch5).max())

print(f"Im(ant2) max absolute value across ALL traces: {max_abs_imag_ant2:.3e}")

THRESHOLD = 1e-10
is_structural_zero = max_abs_imag_ant2 < THRESHOLD

if is_structural_zero:
    print()
    print("VERDICT: Im(ant2) is structurally zero (all values < 1e-10).")
    N_CHANNELS = 5
else:
    print()
    print("VERDICT: Im(ant2) has non-negligible values. Keep all 6 channels.")
    N_CHANNELS = 6

Im(ant2) max absolute value across ALL traces: 0.000e+00

VERDICT: Im(ant2) is structurally zero (all values < 1e-10).


In [6]:
if N_CHANNELS == 5:
    print("Using 5 channels — dropping Im(ant2) (index 5).")
    for r in trace_records:
        r["data"] = r["data"][:, :, :5]  # (P, 234, 5)
elif N_CHANNELS == 6:
    print("Using 6 channels — keeping all.")
else:
    raise ValueError("N_CHANNELS must be 5 or 6.")

sample = trace_records[0]["data"]
print(f"Frame shape after channel selection: {sample.shape} (P, {TARGET_SUBCARRIERS}, {N_CHANNELS})")

Using 5 channels — dropping Im(ant2) (index 5).
Frame shape after channel selection: (361, 234, 5) (P, 234, 5)


---
## 4. Quality Filtering

Remove frames that contain:
- NaN or Inf values (corrupted packets)
- All-zero values (missing / dropped frames)

Done before normalization so bad frames cannot skew the per-trace mean/std.

In [7]:
print("Quality filtering...")
print("-" * 70)

total_kept = 0
total_removed = 0

for r in trace_records:
    features = r["data"]
    P = features.shape[0]

    nan_or_inf = np.all(np.isfinite(features), axis=(1, 2))
    all_zero = np.all(features == 0, axis=(1, 2))
    valid_mask = nan_or_inf & ~all_zero

    clean = features[valid_mask]
    removed = P - clean.shape[0]
    total_removed += removed
    total_kept += clean.shape[0]
    r["data"] = clean

    status = f"  [REMOVED {removed} bad frames]" if removed > 0 else ""
    print(f"  {r['device']:8s} {r['scenario']:12s} {r['stem'][-20:]:22s} "
          f"{P} frames -> {clean.shape[0]} clean{status}")

print("-" * 70)
print(f"Total: {total_kept} frames kept, {total_removed} removed.")

if total_kept == 0:
    raise ValueError("All frames removed! Check your data files.")

Quality filtering...
----------------------------------------------------------------------
  M7       empty        vmatrix_empty-2_M7     361 frames -> 361 clean
  M7       empty        vmatrix_empty-3_M7     361 frames -> 361 clean
  M7       empty        vmatrix_empty_M7       371 frames -> 371 clean
  M7       moving       vmatrix_moving-2_M7    375 frames -> 375 clean
  M7       moving       vmatrix_moving-3_M7    335 frames -> 335 clean
  M7       moving       vmatrix_moving_M7      335 frames -> 335 clean
  M7       stationary   trix_stationary-2_M7   355 frames -> 355 clean
  M7       stationary   trix_stationary-3_M7   366 frames -> 366 clean
  M7       stationary   matrix_stationary_M7   386 frames -> 386 clean
  X7       empty        vmatrix_empty-2_X7     364 frames -> 364 clean
  X7       empty        vmatrix_empty-3_X7     684 frames -> 684 clean
  X7       empty        vmatrix_empty_X7       379 frames -> 379 clean
  X7       moving       vmatrix_moving-2_X7    374 frame

---
## 5. Per-Trace Z-Score Normalization

Each trace is normalized independently using its own mean and std, computed
**per channel** across all frames and subcarriers within that trace.

Formula: `x_norm = (x - μ_c) / σ_c`  where μ_c and σ_c are channel-wise statistics.

- μ and σ computed across frames (axis 0) and subcarriers (axis 1), separately per channel (axis 2)
- This yields a `(1, 1, C)` mean and std tensor applied element-wise
- Zero-variance channels (e.g. structural Im(ant2) if still present) are left as-is (divide by 1)

**Why per-channel, not global?**
Different antenna channels carry signals on different scales. Normalizing per
channel removes both session-level scale differences and inter-channel amplitude
differences, preventing any single channel from disproportionately influencing
the Euclidean distance calculations in ProtoNet.

**Why per-trace, not global?**
Each capture is an independent recording at a different time. A noisier session
should not inflate the std for the quieter ones. Per-trace normalization removes
session-level scale differences before windowing.

In [8]:
print("Applying per-trace, per-channel z-score normalization...")
print("-" * 70)

for r in trace_records:
    features = r["data"]  # (P, K, C)
    mean = features.mean(axis=(0, 1), keepdims=True)  # (1, 1, C)
    std  = features.std( axis=(0, 1), keepdims=True)  # (1, 1, C)
    std  = np.where(std == 0, 1.0, std)               # guard zero-variance

    normalized = (features - mean) / std
    r["data"] = normalized

    post_mean = normalized.mean(axis=(0, 1))
    post_std  = normalized.std( axis=(0, 1))
    print(f"  {r['device']:8s} {r['scenario']:12s} {r['stem'][-20:]:22s}")
    print(f"    mean {', '.join(f'{m:.4f}' for m in post_mean)}")
    print(f"    std  {', '.join(f'{s:.4f}' for s in post_std)}")

print("-" * 70)
print("Normalization complete.")

Applying per-trace, per-channel z-score normalization...
----------------------------------------------------------------------
  M7       empty        vmatrix_empty-2_M7    
    mean 0.0000, -0.0000, 0.0000, -0.0000, -0.0000
    std  1.0000, 1.0000, 1.0000, 1.0000, 1.0000
  M7       empty        vmatrix_empty-3_M7    
    mean 0.0000, 0.0000, 0.0000, -0.0000, 0.0000
    std  1.0000, 1.0000, 1.0000, 1.0000, 1.0000
  M7       empty        vmatrix_empty_M7      
    mean -0.0000, 0.0000, 0.0000, 0.0000, 0.0000
    std  1.0000, 1.0000, 1.0000, 1.0000, 1.0000
  M7       moving       vmatrix_moving-2_M7   
    mean -0.0000, -0.0000, 0.0000, 0.0000, 0.0000
    std  1.0000, 1.0000, 1.0000, 1.0000, 1.0000
  M7       moving       vmatrix_moving-3_M7   
    mean -0.0000, 0.0000, 0.0000, 0.0000, 0.0000
    std  1.0000, 1.0000, 1.0000, 1.0000, 1.0000
  M7       moving       vmatrix_moving_M7     
    mean -0.0000, -0.0000, -0.0000, -0.0000, 0.0000
    std  1.0000, 1.0000, 1.0000, 1.0000, 1.0000
  

---
## 6. Temporal Windowing Per-Trace

Group consecutive frames into non-overlapping windows within each trace.
A window never spans two recording sessions.

- `WINDOW_SIZE = 5` → each window covers 5 consecutive BFI frames
- Each trace yields `P // WINDOW_SIZE` windows, where P varies per trace and device
- There is no global cap: each trace keeps its own number of windows W

Output shape stored in each record: `(W, T, K, C)` where:
- `W` = number of windows in this trace (varies by trace)
- `T` = `WINDOW_SIZE` time steps per window
- `K` = `TARGET_SUBCARRIERS` = 234
- `C` = `N_CHANNELS` = 5 (or 6)

In [9]:
print(f"Temporal windowing (window size = {WINDOW_SIZE} frames)...")
print("-" * 70)

skipped = []
for r in trace_records:
    features = r["data"]  # (P, K, C)
    P = features.shape[0]
    num_windows = P // WINDOW_SIZE

    if num_windows == 0:
        print(f"  WARNING: {r['stem']} only {P} frames, need {WINDOW_SIZE}. Skipping.")
        r["windows"] = None
        skipped.append(r["stem"])
        continue

    usable = num_windows * WINDOW_SIZE
    trimmed = features[:usable]  # (usable, K, C)
    windows = trimmed.reshape(
        num_windows, WINDOW_SIZE,
        TARGET_SUBCARRIERS, N_CHANNELS
    )  # (W, T, K, C)
    r["windows"] = windows

    discarded = P - usable
    note = f"  [{discarded} trailing frames discarded]" if discarded > 0 else ""
    print(f"  {r['device']:8s} {r['scenario']:12s} {r['stem'][-20:]:22s} "
          f"{P} frames -> {num_windows} windows{note}")

if skipped:
    print(f"  {len(skipped)} traces with insufficient frames skipped: {skipped}")

total_windows = sum(r["windows"].shape[0] for r in trace_records if r["windows"] is not None)
print("-" * 70)
print(f"Windowing complete. Total windows across all traces: {total_windows}")
print(f"Each window shape: ({WINDOW_SIZE}, {TARGET_SUBCARRIERS}, {N_CHANNELS}) (T, K, C)")
print()
print("Window counts per device (vary by device BFI rate — expected):")
for dev in DEVICES:
    dev_traces = [r for r in trace_records if r["device"] == dev and r["windows"] is not None]
    counts = [r["windows"].shape[0] for r in dev_traces]
    if counts:
        print(f"  {dev:8s} min={min(counts)} max={max(counts)} traces={len(counts)}")

Temporal windowing (window size = 5 frames)...
----------------------------------------------------------------------
  M7       empty        vmatrix_empty-2_M7     361 frames -> 72 windows  [1 trailing frames discarded]
  M7       empty        vmatrix_empty-3_M7     361 frames -> 72 windows  [1 trailing frames discarded]
  M7       empty        vmatrix_empty_M7       371 frames -> 74 windows  [1 trailing frames discarded]
  M7       moving       vmatrix_moving-2_M7    375 frames -> 75 windows
  M7       moving       vmatrix_moving-3_M7    335 frames -> 67 windows
  M7       moving       vmatrix_moving_M7      335 frames -> 67 windows
  M7       stationary   trix_stationary-2_M7   355 frames -> 71 windows
  M7       stationary   trix_stationary-3_M7   366 frames -> 73 windows  [1 trailing frames discarded]
  M7       stationary   matrix_stationary_M7   386 frames -> 77 windows  [1 trailing frames discarded]
  X7       empty        vmatrix_empty-2_X7     364 frames -> 72 windows  [4 tra

---
## 7. Save Processed Traces

One output file per input trace, named `p_{original_stem}.npy`.

**Why no separate label file?**
The label is deterministic from the filename — any downstream notebook
re-derives it using the same `LABEL_MAP` lookup. This avoids label/data
mismatches and keeps everything self-describing.

Output shape per file: `(W, T, K, C)` = `(W, WINDOW_SIZE, 234, N_CHANNELS)`

W varies across files — cross-device imbalance is handled by the stratified
episode sampler in Notebook 4, not by discarding frames here.

In [10]:
print(f"Saving processed traces to {OUTPUT_DIR} organized by device...")
print("-" * 70)

saved = 0
for r in trace_records:
    if r["windows"] is None:
        print(f"  SKIPPED (no windows): {r['stem']}")
        continue

    device = r["device"]
    device_dir = OUTPUT_DIR / device
    device_dir.mkdir(parents=True, exist_ok=True)

    out_path = device_dir / f"p{r['stem']}.npy"
    np.save(out_path, r["windows"])

    label = LABEL_MAP[r["scenario"]]
    print(f"  {str(device_dir / ('p'+r['stem']+'.npy')):52s} "
          f"{str(r['windows'].shape):20s} label={label} {r['scenario']}")
    saved += 1

print("-" * 70)
print(f"Saved {saved} files under {OUTPUT_DIR}/{{device}}")

Saving processed traces to data\processed organized by device...
----------------------------------------------------------------------
  data\processed\M7\pvmatrix_empty-2_M7.npy            (72, 5, 234, 5)      label=0 empty
  data\processed\M7\pvmatrix_empty-3_M7.npy            (72, 5, 234, 5)      label=0 empty
  data\processed\M7\pvmatrix_empty_M7.npy              (74, 5, 234, 5)      label=0 empty
  data\processed\M7\pvmatrix_moving-2_M7.npy           (75, 5, 234, 5)      label=1 moving
  data\processed\M7\pvmatrix_moving-3_M7.npy           (67, 5, 234, 5)      label=1 moving
  data\processed\M7\pvmatrix_moving_M7.npy             (67, 5, 234, 5)      label=1 moving
  data\processed\M7\pvmatrix_stationary-2_M7.npy       (71, 5, 234, 5)      label=1 stationary
  data\processed\M7\pvmatrix_stationary-3_M7.npy       (73, 5, 234, 5)      label=1 stationary
  data\processed\M7\pvmatrix_stationary_M7.npy         (77, 5, 234, 5)      label=1 stationary
  data\processed\X7\pvmatrix_empty-2

---
## 8. Loader Helper *(copy into Notebooks 2–4)*

Downstream notebooks use this function to load processed traces and assign labels.
It mirrors the same `LABEL_MAP` inference so labels stay consistent.

```python
from pathlib import Path
import numpy as np

LABEL_MAP = {"empty": 0, "stationary": 1, "moving": 1}

def load_processed(processed_dir, devices=None):
    """
    Load all p*.npy files from processed_dir.

    Parameters
    ----------
    processed_dir : str or Path
    devices : list of str or None — filter to specific devices, e.g. ["M7"]

    Returns
    -------
    records : list of dict {windows, label, device, scenario, file}
        Each record holds the (W, T, K, C) array for one trace separately.
        W varies across records — do NOT concatenate before episode sampling.
    """
    processed_dir = Path(processed_dir)
    records = []

    for fpath in sorted(processed_dir.glob("**/p*.npy")):
        stem = fpath.stem[1:]  # strip leading 'p'
        scenario = next((k for k in LABEL_MAP if k in stem.lower()), None)
        if scenario is None:
            continue
        device = next((d for d in ["M7", "X7", "X300"] if stem.endswith(d)), None)
        if devices and device not in devices:
            continue

        windows = np.load(fpath)  # (W, T, K, C)
        records.append({
            "file": fpath.name,
            "device": device,
            "scenario": scenario,
            "label": LABEL_MAP[scenario],
            "windows": windows,   # shape (W, T, K, C), W varies
        })

    return records

# Usage examples:
# All devices — returns list of per-trace dicts
# records = load_processed("data/processed")
#
# Notebook 3 — single device supervised baseline (can concatenate safely)
# records = load_processed("data/processed", devices=["M7"])
# windows = np.concatenate([r["windows"] for r in records], axis=0)
# labels  = np.concatenate([np.full(r["windows"].shape[0], r["label"]) for r in records])
#
# Notebook 4 — ProtoNet stratified sampler uses per-device, per-trace lists directly
# records = load_processed("data/processed")
#
# PyTorch tensor (channels first)
# import torch
# x = torch.from_numpy(windows).float().permute(0, 3, 1, 2)  # (N, C, T, K)
```

---
## 9. Sanity Checks

Reload all saved files and verify correctness.

In [11]:
reloaded = []
for fpath in sorted(OUTPUT_DIR.glob("**/p*.npy")):
    stem = fpath.stem[1:]
    scenario = next((k for k in LABEL_MAP if k in stem.lower()), None)
    device = next((d for d in DEVICES if stem.endswith(d)), None)
    if scenario is None or device is None:
        print(f"  WARNING: Could not parse {fpath.relative_to(OUTPUT_DIR)}")
        continue
    arr = np.load(fpath)
    reloaded.append({"file": fpath.relative_to(OUTPUT_DIR).as_posix(),
                     "device": device, "scenario": scenario, "data": arr})

all_data = np.concatenate([r["data"] for r in reloaded], axis=0)
labels_all = np.array([LABEL_MAP[r["scenario"]]
                        for r in reloaded
                        for _ in range(r["data"].shape[0])], dtype=np.int64)

print("SANITY CHECKS")
print("-" * 70)

has_nan = np.any(np.isnan(all_data))
has_inf = np.any(np.isinf(all_data))
print(f"{'PASS' if not has_nan else 'FAIL'} No NaN values")
print(f"{'PASS' if not has_inf else 'FAIL'} No Inf values")

expected = (WINDOW_SIZE, TARGET_SUBCARRIERS, N_CHANNELS)
shape_ok = all_data.shape[1:] == expected
print(f"{'PASS' if shape_ok else 'FAIL'} Window shape {all_data.shape} "
      f"expected (N, {WINDOW_SIZE}, {TARGET_SUBCARRIERS}, {N_CHANNELS})")

labels_ok = set(np.unique(labels_all)) == {0, 1}
print(f"{'PASS' if labels_ok else 'FAIL'} Labels are binary: {np.unique(labels_all)}")

overall_mean = all_data.mean()
overall_std  = all_data.std()
norm_ok = abs(overall_mean) < 0.5 and 0.3 < overall_std < 3.0
print(f"{'PASS' if norm_ok else 'WARN'} Normalization mean={overall_mean:.4f}, std={overall_std:.4f}")

devices_found = set(r["device"] for r in reloaded)
devices_ok = devices_found == set(DEVICES.keys())
print(f"{'PASS' if devices_ok else 'WARN'} Devices present: {sorted(devices_found)}")

print(f"INFO {len(reloaded)} files saved to {OUTPUT_DIR}")
print("-" * 70)

print("\n--- SUMMARY (copy this into your notes) ---")
print(f"Total windows  : {all_data.shape[0]}")
print(f"Window shape   : ({WINDOW_SIZE}, {TARGET_SUBCARRIERS}, {N_CHANNELS}) (T, K, C)")
print(f"N_CHANNELS     : {N_CHANNELS}")
print(f"No global cap  : window counts vary per trace (expected)")
print(f"Class 0 (empty)   : {(labels_all == 0).sum()} windows")
print(f"Class 1 (occupied): {(labels_all == 1).sum()} windows")
print()
for dev in DEVICES:
    dev_rows = [r for r in reloaded if r["device"] == dev]
    dev_windows = sum(r["data"].shape[0] for r in dev_rows)
    dev_files   = len(dev_rows)
    win_counts  = [r["data"].shape[0] for r in dev_rows]
    print(f"  {dev:8s} {dev_windows} windows across {dev_files} files "
          f"(min={min(win_counts) if win_counts else 0}, max={max(win_counts) if win_counts else 0})")

print()
print("Files in data/processed:")
for r in reloaded:
    label = LABEL_MAP[r["scenario"]]
    print(f"  {r['file']:52s} shape={r['data'].shape} label={label}")

print()
print("PyTorch loading (use in Notebooks 2-4):")
print("  x = torch.from_numpy(windows).float().permute(0, 3, 1, 2)")
print(f"  (N, {WINDOW_SIZE}, {TARGET_SUBCARRIERS}, {N_CHANNELS}) -> "
      f"(N, {N_CHANNELS}, {WINDOW_SIZE}, {TARGET_SUBCARRIERS}) (N, C, T, K)")
print()
print("--- Notebook 1 complete. Proceed to Notebook 2 CNN sanity check. ---")

SANITY CHECKS
----------------------------------------------------------------------
PASS No NaN values
PASS No Inf values
PASS Window shape (5729, 5, 234, 5) expected (N, 5, 234, 5)
PASS Labels are binary: [0 1]
PASS Normalization mean=-0.0000, std=1.0000
PASS Devices present: ['M7', 'X300', 'X7']
INFO 27 files saved to data\processed
----------------------------------------------------------------------

--- SUMMARY (copy this into your notes) ---
Total windows  : 5729
Window shape   : (5, 234, 5) (T, K, C)
N_CHANNELS     : 5
No global cap  : window counts vary per trace (expected)
Class 0 (empty)   : 690 windows
Class 1 (occupied): 5039 windows

  M7       648 windows across 9 files (min=67, max=77)
  X7       4531 windows across 9 files (min=71, max=3631)
  X300     550 windows across 9 files (min=41, max=74)

Files in data/processed:
  M7/pvmatrix_empty-2_M7.npy                           shape=(72, 5, 234, 5) label=0
  M7/pvmatrix_empty-3_M7.npy                           shape=(72